# TPU v5e-1 and GPU T4 tests

## Clone and open the repository

In [1]:
!git clone https://github.com/Kle1n19/RL_JAX_OPTIM_DIPLOMA.git
%cd RL_JAX_OPTIM_DIPLOMA

Cloning into 'RL_JAX_OPTIM_DIPLOMA'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 80 (delta 11), reused 78 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (80/80), 497.28 KiB | 2.33 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/RL_JAX_OPTIM_DIPLOMA


## Let's install the dependencies (and yes, we're doing this manually because JAX has different versions depending on the backend, plus a lot of things are already pre-installed here)

In [2]:
!pip install optuna libcst

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 81.4 MB/s eta 0:00:00


### TPU


In [ ]:
!pip install jax[tpu]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 MB 4.3 MB/s eta 0:00:00
  Attempting uninstall: libtpu
    Found existing installation: libtpu 0.0.21
    Uninstalling libtpu-0.0.21:
      Successfully uninstalled libtpu-0.0.21


### GPU
Verify CUDA version first

In [3]:
!nvidia-smi

Tue Apr 28 16:42:20 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Install with `pip install jax[cuda<version_number>]`

In [ ]:
!pip install -U "jax[cuda13]"

In [6]:
import jax
print(jax.devices())

[CudaDevice(id=0)]


## Experiments
### Convergence Speed

In [ ]:
!python experiments/convergence_speed/run.py --budget 4096 --n-steps 252

ERROR:2026-04-28 16:44:59,841:jax._src.xla_bridge:487: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda13.initialize()
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/xla_bridge.py", line 485, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/usr/local/lib/python3.12/dist-packages/jax_plugins/xla_cuda13/__init__.py", line 334, in initialize
    c_api = xb.register_plugin(
            ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/jax/_src/xla_bridge.py", line 621, in register_plugin
    c_api = xla_client.load_pjrt_plugin_dynamically(plugin_name, library_path)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/jaxlib/xla_client.py", line 113, in load_pjrt_plugin_dynamically
    return _xla.load_pjrt_plugin(plugin_name, library_path, c_api=None)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

### Speedup Magnitude

In [11]:
!python experiments/speedup_magnitude/run.py --budget 1000 --n-steps 252

━━━━              Exp — Speedup Magnitude               ━━━━
  Device : GPU  —  GPU
  JAX    : 0.7.2  |  1 device(s) visible
  Domain: American Option  [MC]
  TPE  (budget=1000)

  → best reward=3.763  best speedup=3.763x  precision_ok=100%  (265.0s)
  Domain: European Option  [MC]
  TPE  (budget=1000)

  → best reward=4.281  best speedup=4.281x  precision_ok=100%  (269.5s)
  Domain: Basket Option  [MC]
  TPE  (budget=1000)

  → best reward=6.964  best speedup=6.964x  precision_ok=100%  (283.1s)
  Domain: Runge-Kutta ODE  [Non-MC]
  TPE  (budget=1000)

  → best reward=2.611  best speedup=2.611x  precision_ok=100%  (66.6s)
  Domain: Kalman Smoother  [Non-MC]
  TPE  (budget=1000)
/usr/local/lib/python3.12/dist-packages/jax/_src/interpreters/mlir.py:1334: UserWarning: Some donated buffers were not usable: uint32[1024,2].
See an explanation at https://docs.jax.dev/en/latest/faq.html#buffer-donation.
  warnings.warn("Some donated buffers were not usable:"

  → best reward=1.542  best speedu

### Distributional Integrity

In [12]:
!python experiments/distributional_integrity/run.py

━━━━           Exp — Distributional Integrity           ━━━━
  Device : GPU  —  GPU
  JAX    : 0.7.2  |  1 device(s) visible
Finding best tuned params…
  Using cached best_params from speedup_magnitude run.
  best_params: {'scan_unroll': 16, 'scan_reverse': True, 'jit_donate_argnums': '()', 'dot_precision': 'highest', 'matmul_precision': 'high', 'autotune_level': 0, 'map_chunk_size': 4, 'checkpoint_policy': None}


  W1 distance:       0.0649
  Max quantile drift: 0.0091
  Mean quantile drift:0.0018

Results saved  /content/RL_JAX_OPTIM_DIPLOMA/experiments/distributional_integrity/results/results.json

Generating plots…
  saved: /content/RL_JAX_OPTIM_DIPLOMA/experiments/distributional_integrity/figures/01_distributional_integrity.png

Done. Figures in /content/RL_JAX_OPTIM_DIPLOMA/experiments/distributional_integrity/figures/


## Getting traces with the best params for future analysis

In [ ]:
!python get_trace.py

/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0424 00:16:40.744611   12476 python_hooks.cc:416] Can't import tensorflow.python.profiler.trace
E0424 00:16:40.768606   12476 python_hooks.cc:416] Can't import tensorflow.python.profiler.trace
Perfetto trace saved → ./traces/baseline
Open at https://ui.perfetto.dev
[ep    1/60] [OK ]  speedup=1.35x  reward=   1.353  scan_unroll=4  scan_reverse=True  jit_donate_argnums=(1,)  dot_precision=high  matmul_precision=high  autotune_level=3  map_chunk_size=8  checkpoint_policy=<function dots_with_no_batch_dims_saveable at 0x7ccea1324ea0>
[ep    2/60] [OK ]  speedup=0.98x  reward=   0.983  scan_unroll=16  scan_re

In [ ]:
!python analyze_traces.py

## Here you can download the reults(traces, JSON's and plots)

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("t4_results", "zip", ".", "experiments/speedup_magnitude")
shutil.make_archive("t4_traces", "zip", ".", "traces")
shutil.make_archive("t4_trace_analysis","zip", ".", "results/analysis_gpu.txt")
shutil.make_archive("t4_results_speed", "zip", ".", "experiments/convergence_speed")
shutil.make_archive("traces_t4",  "zip", ".", "traces")
files.download("t4_results_speed.zip")
files.download("t4_results.zip")
files.download("t4_traces.zip")
files.download("t4_trace_analysis.zip")
files.download("traces_t4.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>